# 80 Judge training (A100)

Trains the judge: the same Qwen2.5-Coder-7B as the generator with a second LoRA adapter (started as a copy of the generator's, `rl_v1_it8`) and a small cost head. For each candidate query it learns two things: **is it correct** ("yes" or "no") and **how costly is it** compared with the question's other correct answers. Training data: `judge_data_v1` (55,904 already executed candidates for 8,141 train questions: the generator's own answers and the pool), validated on the 4 held-out databases (579 questions).

One pass, 473 optimizer steps, about **1.5 hours on an A100 (about 18 units)**. The first step holds the longest sequence, so a memory problem shows within a minute. Progress is logged every step (loss, accuracy, tokens per second, projected total minutes); validation metrics at steps 200 and 400 and at the end.

**Resumable:** after a disconnect, run the setup, install and config cells again, then the training cell: it continues from the last checkpoint (at most 50 steps lost).

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
import json, os
R = f"{DRIVE_ROOT}/effsql/runs"
DATA = "judge_data_v1"    # the judge's training data (built from own_v1 and pool_v1, already on Drive)
JUDGE = "judge_v1"        # the judge: adapter + cost head
CAND = "judge_cand_v1"    # candidates, scores and labels for val, dev and mini-dev
SEL = "judge_v1_sel"      # the selected answers (also judge_v1_sel_p and judge_v1_sel_vote)
GEN = "rl_v1_it8"         # the generator
VAL_DBS = ",".join(json.load(open(f"{R}/sft_v1/export_meta.json"))["holdout_db_ids"])
assert os.path.exists(f"{R}/{DATA}/data_meta.json"), f"{DATA} is missing on Drive"
assert json.load(open(f"{R}/{GEN}/adapter_meta.json"))["complete"]
print(json.dumps(json.load(open(f"{R}/{DATA}/data_meta.json"))["stats"], indent=1))
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. Train (about 1.5 hours)

In [ ]:
!python -m effsql.judge train --run {JUDGE} --data {DATA}

### 2. Validation on the held-out databases

`auc`: how well p separates correct from wrong candidates (0.5 is chance, 1 is perfect). `top1`: among questions with at least one correct candidate, how often the candidate with the highest p is correct. `cost_pair_acc`: for two correct candidates of one question whose costs differ by at least 1.4x, how often c orders them right (0.5 is chance).

In [ ]:
for line in open(f"{R}/{JUDGE}/val_log.jsonl"):
    v = json.loads(line)
    print({k: (round(x, 3) if isinstance(x, float) else x) for k, x in v.items()})

### Done: disconnect now, then run `81_judge_candidates` (A100)

*Runtime > Disconnect and delete runtime.* An idle GPU keeps using compute units.